<a href="https://colab.research.google.com/github/ashnaimran06/my-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ashnaimran06/my-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

For my Refresh / Content Opportunity Scoring lane, one row in the daily performance table represents one **content item for one client on one report date**.

I will use **March 2026 (`month=2026-03`)** as my development month. I chose a mid-panel month so that I can develop my features and label logic without using the final June 2026 month.


In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

!pip -q install duckdb

In [5]:
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

print("Token loaded:", HF_TOKEN is not None)

Token loaded: True


In [6]:
import duckdb

con = duckdb.connect()

con.execute(
    f"CREATE SECRET (TYPE huggingface, TOKEN '{HF_TOKEN}')"
)

print("Warehouse connection ready!")

Warehouse connection ready!


In [7]:
query = """
SELECT
    COUNT(*) AS row_count,
    MIN(report_date) AS min_date,
    MAX(report_date) AS max_date
FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet'
)
WHERE month = '2026-03'
"""

con.sql(query)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌───────────┬────────────┬────────────┐
│ row_count │  min_date  │  max_date  │
│   int64   │    date    │    date    │
├───────────┼────────────┼────────────┤
│   9841378 │ 2026-03-01 │ 2026-03-31 │
└───────────┴────────────┴────────────┘

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

### Fields: feature / label / context / excluded

**Features**

* `gsc_impressions` — shows how much search visibility a content item receives.
* `gsc_clicks` — shows how much search traffic it receives.
* `gsc_sum_position` — provides information about search ranking performance.
* `sessions_ai` — shows AI-referred sessions and provides additional traffic context.
* `scroll_events` — provides a signal of content engagement.

**Label**

* `is_declining_label` — a proxy label for whether a content item is showing declining performance. It will be created separately and will not be used as a feature.

**Context**

* `client_hash_id` — identifies the client for grouping, but is not used as a predictive feature.
* `content_hash_id` — identifies the content item for grouping and joining, but is not used as a predictive feature.
* `report_date` — identifies when the performance was observed.
* `month` — identifies the warehouse month/partition.
* Data-availability fields such as `client_has_gsc`, `client_has_ga4`, `gsc_data_available`, and `ga4_data_available` — describe whether relevant data sources are available.

**Excluded**

* `client_hash_id` and `content_hash_id` are excluded from features because they are identifiers rather than meaningful predictive signals.
* Any field used to construct the declining label is excluded from features because it would leak the answer into the model.
* Any precomputed composite score or recommendation field is excluded because I want to build the scoring from underlying signals rather than blindly trust an existing decision.


In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


query = """
DESCRIBE SELECT *
FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet'
)
LIMIT 1
"""

con.sql(query)

┌────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│    column_name     │ column_type │  null   │   key   │ default │  extra  │
│      varchar       │   varchar   │ varchar │ varchar │ varchar │ varchar │
├────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ report_date        │ DATE        │ YES     │ NULL    │ NULL    │ NULL    │
│ client_hash_id     │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ content_hash_id    │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ client_has_gsc     │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ client_has_ga4     │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_data_available │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ ga4_data_available │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_impressions    │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_clicks         │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# GRAIN

query = """
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT (report_date, client_hash_id, content_hash_id)) AS distinct_date_client_content_keys,
    MAX(rows_per_key) AS max_rows_per_key
FROM (
    SELECT
        report_date,
        client_hash_id,
        content_hash_id,
        COUNT(*) AS rows_per_key
    FROM read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet'
    )
    WHERE month = '2026-03'
    GROUP BY report_date, client_hash_id, content_hash_id
)
"""

con.sql(query)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────┬───────────────────────────────────┬──────────────────┐
│ total_rows │ distinct_date_client_content_keys │ max_rows_per_key │
│   int64    │               int64               │      int64       │
├────────────┼───────────────────────────────────┼──────────────────┤
│    9841378 │                           9841378 │                1 │
└────────────┴───────────────────────────────────┴──────────────────┘

In [10]:
# Verify row count + date window

query = """
SELECT
    COUNT(*) AS row_count,
    MIN(report_date) AS min_date,
    MAX(report_date) AS max_date
FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet'
)
WHERE month = '2026-03'
"""

con.sql(query)

┌───────────┬────────────┬────────────┐
│ row_count │  min_date  │  max_date  │
│   int64   │    date    │    date    │
├───────────┼────────────┼────────────┤
│   9841378 │ 2026-03-01 │ 2026-03-31 │
└───────────┴────────────┴────────────┘

In [11]:
# Verify availability

query = """
SELECT
    COUNT(*) AS march_rows,
    COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS gsc_available_rows,
    COUNT(*) FILTER (WHERE gsc_data_available IS TRUE)
        * 100.0 / COUNT(*) AS gsc_available_pct
FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet'
)
WHERE month = '2026-03'
"""

con.sql(query)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────┬────────────────────┬───────────────────┐
│ march_rows │ gsc_available_rows │ gsc_available_pct │
│   int64    │       int64        │      double       │
├────────────┼────────────────────┼───────────────────┤
│    9841378 │            3611061 │ 36.69263592964319 │
└────────────┴────────────────────┴───────────────────┘

In March 2026, 3,611,061 of 9,841,378 rows (36.69%) had GSC data available.

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*


- This data cannot tell me the true reason why a page is declining or whether a refresh will definitely improve its future performance.

The history is also unbalanced across clients and content items, so some pages have much more historical data than others. Early rows may be GSC-only because different data sources are not available for every client or period. In addition, rolling performance windows can overlap, so observations from nearby dates are not fully independent.

These limitations mean my score should be treated as **decision support for prioritizing pages to review**, not as proof that a specific page will improve after a refresh.


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.